<a href="https://colab.research.google.com/github/jayaprakash-kolla/MTechThesis-EEG2TexT/blob/main/Preprocessing_with_ICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mne
!pip install mne-bids
!pip install tensorflow

In [3]:
import os
import mne
from mne.io import read_raw_edf
from mne.preprocessing import ICA
import mne_bids
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
# Note: ASR import has been removed as it's not in your function

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# --- HELPER FUNCTIONS ---

def preprocess_raw(raw):
    """
    Applies optimized preprocessing with ICA for artifact correction:
    1. Sets EOG types
    2. High-pass filters (0.5 Hz)
    3. Fits ICA on a 1.0 Hz filtered copy
    4. Finds and removes EOG (blink) components
    5. Removes the 129th channel
    6. Applies Common Average Reference (CAR)
    7. Downsamples to 250 Hz
    (No low-pass filter is applied)
    """

    # CRITICAL: Ensure data is loaded into memory
    raw.load_data()

    # 1. Handle EOG Channels (Based on previous warnings)
    eog_channels = ['E8', 'E26', 'E126', 'E127']
    raw.set_channel_types({ch: 'eog' for ch in eog_channels})

    # 2. High-pass filter the main data at 0.5 Hz
    raw.filter(l_freq=0.5, h_freq=None, fir_design='firwin', verbose=False)

    # 3. Fit ICA on a 1.0 Hz filtered copy
    print("    ... Fitting ICA on 1.0 Hz filtered data")
    raw_for_ica = raw.copy().filter(l_freq=1.0, h_freq=None, fir_design='firwin', verbose=False)

    ica = ICA(n_components=30, random_state=42, max_iter='auto')
    ica.fit(raw_for_ica, verbose=False)

    # 4. Find and remove EOG (blink) components
    print("    ... Finding EOG components")
    eog_indices, eog_scores = ica.find_bads_eog(raw_for_ica, ch_name=eog_channels)

    if eog_indices:
        print(f"    ... Found {len(eog_indices)} EOG components: {eog_indices}")
        ica.exclude = eog_indices

        # Apply the ICA solution to the *original* 0.5 Hz filtered data
        print("    ... Removing EOG components")
        ica.apply(raw, verbose=False)
    else:
        print("    ... No EOG components found.")

    # 5. Low-pass filter (REMOVED)
    # raw.filter(l_freq=None, h_freq=40.0, fir_design='firwin', verbose=False)

    # 6. Remove the 129th channel
    ref_channel_name = raw.ch_names[-1]
    print(f"    ... Dropping channel: {ref_channel_name}")
    raw.drop_channels([ref_channel_name])

    # 7. Apply Common Average Reference (CAR)
    print("    ... Applying Common Average Reference (CAR)")
    raw.set_eeg_reference('average', projection=False, verbose=False)

    # 8. Downsample to 250 Hz
    # WARNING: Downsampling without a prior low-pass filter
    # (below 125 Hz) will cause aliasing artifacts.
    print(f"    ... Downsampling from {raw.info['sfreq']} Hz to 250 Hz")
    raw.resample(sfreq=300, verbose=False)

    return raw


In [6]:
def label_and_epoch_data(raw, subject_id, session_id):
    """Performs cross-event linkage for RECALL only and creates X and Y arrays."""

    full_metadata = raw.annotations.to_data_frame()
    full_metadata['item_name'] = full_metadata['item_name'].astype(str)
    full_metadata['session'] = session_id
    full_metadata['subject'] = subject_id

    recalled_words_set = set(full_metadata[full_metadata['description'] == 'REC_WORD']['item_name'].unique())
    recalled_words_set.discard('nan'); recalled_words_set.discard('n/a')

    word_annots_df = full_metadata[full_metadata['description'] == 'WORD'].reset_index(drop=True)
    word_annots_df['is_recalled'] = word_annots_df['item_name'].apply(
        lambda x: 1 if x in recalled_words_set else 0
    )

    word_event_id = {'WORD': 1}
    events, _ = mne.events_from_annotations(raw, event_id=word_event_id, verbose=False)

    Y_df = word_annots_df[[
        'onset', 'item_name', 'subject', 'session', 'is_recalled'
    ]].copy()

    epochs = mne.Epochs(
        raw,
        events,
        event_id=word_event_id,
        tmin=-0.2, tmax=1.0,
        preload=True,
        baseline=(-0.2, 0),
        verbose=False
    )
    X = epochs.get_data()

    if len(X) != len(Y_df):
        raise RuntimeError(f"X and Y arrays are out of sync! X={len(X)}, Y={len(Y_df)}")

    return X, Y_df


In [ ]:
# --- CONFIGURATION ---
bids_root = r"/content/drive/MyDrive/ds004395" # Update this path to where your dataset is located in Colab
'''subject_id = 'LTP324'''
task = 'ltpFR2'
'''session_id = '8'''

# Define a new output directory for your clean data
output_dir = r"/content/drive/MyDrive/PEERS/ds004395/preprocessed_ica/"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory set to: {output_dir}")

all_subjects = ['LTP324']
''' --------  ['LTP063', 'LTP064', 'LTP065', 'LTP066', 'LTP067', 'LTP068', 'LTP069', 'LTP070']  ----------'''
all_sessions = ['8']
''' --------  ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19'] # Add all session IDs here  ----------'''



# --- MAIN LOOP (Nested) ---

# Outer loop for subjects
for subject_id in all_subjects:
    print(f"Processing Subject: {subject_id}")
    # These lists will hold data *only for this subject*
    subject_X_data = []
    subject_Y_data = []

    # Inner loop for sessions
    for session_id in all_sessions:
        print(f"\n--- Processing Session {session_id} ---")

        try:
            bids_path = mne_bids.BIDSPath(
                subject=subject_id, session=session_id, datatype='eeg',
                task=task, root=bids_root
            )
            raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': True})
            print(f"    Loaded raw data: {raw.info['nchan']} channels, {raw.info['sfreq']} Hz.")
            print(f"    Channels found in raw data after loading: {raw.ch_names}") # <-- Added this line for debugging

            raw = preprocess_raw(raw)

            X_session, Y_session = label_and_epoch_data(raw, subject_id, session_id)

            # Add this session's data to the subject's lists
            subject_X_data.append(X_session)
            subject_Y_data.append(Y_session)

            print(f"    Session {session_id} successful. Extracted {len(X_session)} epochs.")

        except FileNotFoundError:
            print(f"❌ File not found for Subject {subject_id}, Session {session_id}. Skipping.")
        except Exception as e:
            print(f"❌ An error occurred for Subject {subject_id}, Session {session_id}: {e}")


Output directory set to: /content/drive/MyDrive/PEERS/ds004395/preprocessed_ica/
Processing Subject: LTP324

--- Processing Session 8 ---
Extracting EDF parameters from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_eeg.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2942499  =      0.000 ...  5884.998 secs...
Reading events from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_events.tsv.
Reading channel info from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_task-ltpFR2_channels.tsv.
Reading electrode coords from /content/drive/MyDrive/ds004395/sub-LTP324/ses-8/eeg/sub-LTP324_ses-8_space-CapTrak_electrodes.tsv.
Not fully anonymizing info - keeping his_id, sex, and hand info
    Loaded raw data: 129 channels, 500.0 Hz.
    Channels found in raw data after loading: ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15

/tmp/ipython-input-1426748282.py:37: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['E8', 'E25', 'E126', 'E127']
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': True})
/tmp/ipython-input-1426748282.py:37: RuntimeWarning: DigMontage is only a subset of info. There is 1 channel position not present in the DigMontage. The channel missing from the montage is:

['E129'].

Consider using inst.rename_channels to match the montage nomenclature, or inst.set_channel_types if this is not an EEG channel, or use the on_missing parameter if the channel position is allowed to be unknown in your analyses.
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=True, extra_params={'preload': True})
/tmp/ipython-input-1426748282.py:37: RuntimeWarning: Not setting positions of 4 eog channels found in montage:
['E8', 'E25', 'E126', 'E127']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.s

    ... Fitting ICA on 1.0 Hz filtered data


In [ ]:
# --- NEW: PROCESS AND SAVE DATA FOR THIS SUBJECT ---
if subject_X_data:
    # 1. Combine all sessions for this subject
    X_combined = np.concatenate(subject_X_data, axis=0)
    Y_combined = pd.concat(subject_Y_data, ignore_index=True)

    print(f"\n--- Processing data for Subject {subject_id} ---")

    # 2. Define Target Label
    target_label = 'is_recalled'
    y = Y_combined[target_label].values

    # 3. Split into Training and Testing sets (per subject)
    X_train, X_test, y_train, y_test = train_test_split(
        X_combined, y, test_size=0.2, stratify=y, random_state=42
    )

    # 4. Apply RobustScaler
    scaler = RobustScaler()
    n_trials_train, n_chans, n_times = X_train.shape
    X_train_reshaped = X_train.reshape(n_trials_train * n_chans, n_times)

    print("    Fitting RobustScaler...")
    scaler.fit(X_train_reshaped)

    # Transform training data
    X_train_scaled_reshaped = scaler.transform(X_train_reshaped)
    X_train_scaled = X_train_scaled_reshaped.reshape(n_trials_train, n_chans, n_times)

    # Transform test data
    n_trials_test = X_test.shape[0]
    X_test_reshaped = X_test.reshape(n_trials_test * n_chans, n_times)
    X_test_scaled_reshaped = scaler.transform(X_test_reshaped)
    X_test_scaled = X_test_scaled_reshaped.reshape(n_trials_test, n_chans, n_times)

    print("Data standardized.")

    # 5. SAVE FINAL DATA (Your requested format)
    output_filename = os.path.join(output_dir, f"{subject_id}_scaled_data.npz")

    print(f"Saving scaled data to {output_filename}...")

    np.savez_compressed(
        output_filename,
        X_train=X_train_scaled,
        X_test=X_test_scaled,
        y_train=y_train,
        y_test=y_test
    )

    print(f"✅ Saved {subject_id} data.")

else:
    print(f"--- No data processed for Subject {subject_id}. No files saved. ---")

print("All subjects have been preprocessed and saved!")